# Visual Entropy vs. Lightweight CNNs: Colab Experiment Runner

This notebook allows running training and sweeps on Google Colab GPU/TPU environments.

> **Important Methodological Note (AGENTS.md):**
> Hardware latency numbers measured in Google Colab (e.g., T4/V100/A100) are **not comparable** with the local embedded/desktop target (Intel i7-10510U / NVIDIA GTX 1050 Pascal). For publication tables, latency must be benchmarked on the local reference machine. All training runs executed here log the active device name and environment specifications.

### Pipeline Principles:
1. Reuses the exact same Python modular code: `python -m experiments.evaluate_cnn`.
2. Strict reproducibility using seeds `[42, 43, 44]`.
3. Identical test partitions as classical ML baselines (task_06A).

In [1]:
# 1. Environment and Hardware Verification
import os
import sys
import torch
import platform

print(f"Python: {platform.python_version()}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")
    print(f"Compute Capability: {torch.cuda.get_device_capability(0)}")
else:
    print("Running on CPU mode")

Python: 3.13.18
PyTorch: 2.14.1+cpu
CUDA Available: False
Running on CPU mode


In [ ]:
# 2. Mount Google Drive or Locate Repository Root
try:
    from google.colab import drive
    drive.mount('/content/drive')
    # Adjust this path to match your Google Drive folder structure
    REPO_DIR = '/content/drive/MyDrive/2026-10-Symmetry-Entropy/code'
    if os.path.isdir(REPO_DIR):
        os.chdir(REPO_DIR)
        print(f"Changed working directory to: {os.getcwd()}")
    else:
        print(f"Warning: {REPO_DIR} not found. Please verify Drive path.")
except ImportError:
    print("Local Jupyter execution detected; keeping current directory.")

In [ ]:
# 3. Dry-Run Validation (Zero training cost)
!python -m experiments.evaluate_cnn --config configs/cnn/colab.toml --dry-run
!python -m experiments.evaluate_cnn --config configs/cnn/mvtec_ad.toml --object bottle --dry-run

In [ ]:
# 4. KolektorSDD2: Train and Benchmark Lightweight CNN Architectures
# Standard polarity
!python -m experiments.evaluate_cnn --config configs/cnn/colab.toml --model ds_cnn
!python -m experiments.evaluate_cnn --config configs/cnn/colab.toml --model tiny_cnn
!python -m experiments.evaluate_cnn --config configs/cnn/colab.toml --model tiny_resnet

# Inverted polarity (255 - x) to evaluate illumination invariance
!python -m experiments.evaluate_cnn --config configs/cnn/colab.toml --model ds_cnn --inverted

In [ ]:
# 5. MVTec-AD: Run Multi-Object Sweep on Representative Objects
for obj in ["bottle", "metal_nut", "screw", "tile"]:
    print(f"=== Evaluating MVTec-AD: {obj} ===")
    !python -m experiments.evaluate_cnn --config configs/cnn/mvtec_ad.toml --object {obj} --model ds_cnn --device cuda

In [ ]:
# 6. Cross-Paradigm Comparison (x94 Classical ML vs. Lightweight CNNs)
!python -m experiments.compare_baseline_vs_cnn --dataset kolektorsdd2
!python -m experiments.compare_baseline_vs_cnn --dataset kolektorsdd2 --inverted
!python -m experiments.compare_baseline_vs_cnn --dataset mvtecad --object bottle

In [ ]:
# 7. Archive and Download Generated Results
import shutil

archive_name = 'cnn_results_colab'
if os.path.isdir('results/cnn'):
    shutil.make_archive(archive_name, 'zip', 'results/cnn')
    print(f"Archived results/cnn to {archive_name}.zip")
    try:
        from google.colab import files
        files.download(f"{archive_name}.zip")
    except ImportError:
        pass